In [ ]:
# ============================================================
# Clean baseline setup
# Load, validate, standardize, and sample the original dataset
# ============================================================

from pathlib import Path

import joblib
import numpy as np
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SEED = 42
#SAMPLE_SEED = 7
SAMPLE_SIZE = 300_000

BASELINE_RESULTS_DIR = Path(
    "02_Baseline_Results_Seed_42"
)

BASELINE_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Load original data
# ------------------------------------------------------------

X = np.load(
    "X.npy",
    mmap_mode=None
)

y = np.load(
    "y.npy",
    mmap_mode=None
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X dtype:", X.dtype)
print("y dtype:", y.dtype)

# ------------------------------------------------------------
# Validate original data
# ------------------------------------------------------------

if X.ndim != 2:
    raise ValueError(
        f"X must be a two-dimensional matrix. Received: {X.shape}"
    )

if len(X) != len(y):
    raise ValueError(
        "X and y have different numbers of samples: "
        f"{len(X):,} and {len(y):,}"
    )

if not np.isfinite(X).all():
    raise ValueError(
        "X contains NaN or infinite values."
    )

print("Any NaN in X:", np.isnan(X).any())
print("Any Inf in X:", np.isinf(X).any())

n_samples, n_features = X.shape

# ------------------------------------------------------------
# Standardize the complete original feature matrix
# ------------------------------------------------------------

scaler = StandardScaler()

X_full = scaler.fit_transform(
    X
).astype(
    np.float32,
    copy=False
)

joblib.dump(
    scaler,
    BASELINE_RESULTS_DIR
    / "Baseline_scaler.pkl"
)

print("Scaled shape:", X_full.shape)
print("Scaled dtype:", X_full.dtype)

# ------------------------------------------------------------
# Create a fixed fitting subset
# ------------------------------------------------------------

actual_sample_size = min(
    SAMPLE_SIZE,
    n_samples
)

sample_rng = np.random.default_rng(
    SEED
)

sample_indices = sample_rng.choice(
    n_samples,
    size=actual_sample_size,
    replace=False
)

sample_indices = np.asarray(
    sample_indices,
    dtype=np.int64
)

X_sample = X_full[
    sample_indices
]

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_sample_indices.npy",
    sample_indices
)

print("Sample shape:", X_sample.shape)
print(
    "Sample percentage:",
    f"{100 * len(X_sample) / n_samples:.4f}%"
)

print(
    "Baseline results folder:",
    BASELINE_RESULTS_DIR.resolve()
)

In [ ]:
# ============================================================
# Exact top-fraction anomaly selection
# ============================================================

def select_exact_top_fraction(
    scores,
    fraction=0.02
):
    """
    Select exactly ceil(fraction * number of samples)
    observations having the highest anomaly scores.

    Higher scores must indicate greater anomaly likelihood.
    """

    scores = np.asarray(
        scores,
        dtype=np.float64
    ).reshape(-1)

    if scores.size == 0:
        raise ValueError(
            "The score array is empty."
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            "The score array contains NaN or infinite values."
        )

    if not 0 < fraction <= 1:
        raise ValueError(
            "fraction must be in the interval (0, 1]."
        )

    selected_count = int(
        np.ceil(
            fraction * scores.size
        )
    )

    selected_count = min(
        selected_count,
        scores.size
    )

    selected_indices = np.argpartition(
        scores,
        scores.size - selected_count
    )[
        scores.size - selected_count:
    ]

    anomaly_labels = np.zeros(
        scores.size,
        dtype=np.uint8
    )

    anomaly_labels[
        selected_indices
    ] = 1

    threshold = float(
        np.min(
            scores[selected_indices]
        )
    )

    return (
        anomaly_labels,
        threshold,
        selected_indices
    )

In [ ]:
# ============================================================
# Clean Isolation Forest baseline
# ============================================================

from sklearn.ensemble import IsolationForest

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TOP_FRACTION = 0.02
BATCH_SIZE = 100_000

# ------------------------------------------------------------
# Train Isolation Forest on the clean fitting subset
# ------------------------------------------------------------

if_model = IsolationForest(
    n_estimators=100,
    max_samples=256,
    contamination=0.02,
    random_state=SEED,
    n_jobs=-1
)

if_model.fit(
    X_sample
)

joblib.dump(
    if_model,
    BASELINE_RESULTS_DIR
    / "Baseline_IF_model.pkl"
)

print("Isolation Forest trained.")

# ------------------------------------------------------------
# Compute anomaly scores for the complete clean dataset
# ------------------------------------------------------------

def compute_if_scores(
    model,
    X_data,
    batch_size=100_000
):
    """
    Compute Isolation Forest scores in batches.

    The sign is reversed so that larger values indicate
    greater anomaly likelihood.
    """

    score_batches = []

    for start_idx in range(
        0,
        len(X_data),
        batch_size
    ):
        end_idx = min(
            start_idx + batch_size,
            len(X_data)
        )

        batch = np.asarray(
            X_data[start_idx:end_idx],
            dtype=np.float32
        )

        batch_scores = -model.decision_function(
            batch
        )

        score_batches.append(
            batch_scores.astype(
                np.float32,
                copy=False
            )
        )

    return np.concatenate(
        score_batches
    )


IF_scores_raw = compute_if_scores(
    model=if_model,
    X_data=X_full,
    batch_size=BATCH_SIZE
)

if IF_scores_raw.shape != (n_samples,):
    raise ValueError(
        "Unexpected Isolation Forest score shape: "
        f"{IF_scores_raw.shape}"
    )

if not np.isfinite(
    IF_scores_raw
).all():
    raise ValueError(
        "Isolation Forest scores contain NaN or Inf."
    )

print(
    "IF scores computed:",
    IF_scores_raw.shape
)

# ------------------------------------------------------------
# Robust score normalization
# ------------------------------------------------------------

if_p1, if_p99 = np.percentile(
    IF_scores_raw,
    [1, 99]
)

if_p1 = float(if_p1)
if_p99 = float(if_p99)

if if_p99 <= if_p1:
    raise ValueError(
        "Invalid Isolation Forest normalization bounds: "
        f"p1={if_p1}, p99={if_p99}"
    )

IF_scores_clipped = np.clip(
    IF_scores_raw,
    if_p1,
    if_p99
)

IF_norm = (
    IF_scores_clipped - if_p1
) / (
    if_p99 - if_p1 + 1e-8
)

IF_norm = IF_norm.astype(
    np.float32,
    copy=False
)

# ------------------------------------------------------------
# Select exactly the highest-scoring 2%
# ------------------------------------------------------------

(
    IF_anomalies,
    IF_threshold,
    IF_selected_indices
) = select_exact_top_fraction(
    scores=IF_norm,
    fraction=TOP_FRACTION
)

selected_count = int(
    IF_anomalies.sum()
)

expected_count = int(
    np.ceil(
        TOP_FRACTION * n_samples
    )
)

assert selected_count == expected_count

# ------------------------------------------------------------
# Save clean Isolation Forest results
# ------------------------------------------------------------

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_IF_scores_raw.npy",
    IF_scores_raw
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_IF_scores_norm.npy",
    IF_norm
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_IF_anomaly_labels.npy",
    IF_anomalies
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_IF_selected_indices.npy",
    IF_selected_indices.astype(
        np.int64
    )
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_IF_normalization_params.npz",
    p1=np.float64(if_p1),
    p99=np.float64(if_p99)
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_IF_selection_params.npz",
    fraction=np.float64(
        TOP_FRACTION
    ),
    selected_count=np.int64(
        selected_count
    ),
    threshold=np.float64(
        IF_threshold
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nIsolation Forest baseline completed.")
print(
    "Raw score range:",
    f"{IF_scores_raw.min():.6f} to "
    f"{IF_scores_raw.max():.6f}"
)

print(
    "Normalization bounds:",
    f"p1={if_p1:.6f}, "
    f"p99={if_p99:.6f}"
)

print(
    "Selected anomalies:",
    f"{selected_count:,}"
)

print(
    "Selected percentage:",
    f"{100 * IF_anomalies.mean():.6f}%"
)

print(
    "Selection threshold:",
    f"{IF_threshold:.8f}"
)

In [ ]:
# ============================================================
# Clean Local Outlier Factor baseline
# ============================================================

from sklearn.neighbors import LocalOutlierFactor
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

LOF_TOP_FRACTION = 0.02
LOF_BATCH_SIZE = 100_000

# ------------------------------------------------------------
# Train LOF on the same clean fitting subset
# ------------------------------------------------------------

lof_model = LocalOutlierFactor(
    n_neighbors=20,
    novelty=True,
    n_jobs=-1
)

lof_model.fit(
    X_sample
)

joblib.dump(
    lof_model,
    BASELINE_RESULTS_DIR
    / "Baseline_LOF_model.pkl"
)

print("LOF trained and saved.")

# ------------------------------------------------------------
# Compute LOF scores for the complete clean dataset
# ------------------------------------------------------------

def compute_lof_scores(
    model,
    X_data,
    batch_size=100_000
):
    """
    Compute LOF anomaly scores in batches.

    Higher returned values indicate greater anomaly likelihood.
    """

    score_batches = []

    total_batches = int(
        np.ceil(
            len(X_data) / batch_size
        )
    )

    for start_idx in tqdm(
        range(0, len(X_data), batch_size),
        total=total_batches,
        desc="Computing LOF scores",
        unit="batch"
    ):
        end_idx = min(
            start_idx + batch_size,
            len(X_data)
        )

        batch = np.asarray(
            X_data[start_idx:end_idx],
            dtype=np.float32
        )

        # score_samples:
        # larger values indicate greater normality.
        # Negation makes larger values more anomalous.
        batch_scores = -model.score_samples(
            batch
        )

        score_batches.append(
            batch_scores.astype(
                np.float32,
                copy=False
            )
        )

    return np.concatenate(
        score_batches
    )


LOF_scores_raw = compute_lof_scores(
    model=lof_model,
    X_data=X_full,
    batch_size=LOF_BATCH_SIZE
)

if LOF_scores_raw.shape != (n_samples,):
    raise ValueError(
        "Unexpected LOF score shape: "
        f"{LOF_scores_raw.shape}"
    )

if not np.isfinite(
    LOF_scores_raw
).all():
    raise ValueError(
        "LOF scores contain NaN or infinite values."
    )

print(
    "LOF scores computed:",
    LOF_scores_raw.shape
)

# ------------------------------------------------------------
# Robust score normalization
# ------------------------------------------------------------

lof_p1, lof_p99 = np.percentile(
    LOF_scores_raw,
    [1, 99]
)

lof_p1 = float(lof_p1)
lof_p99 = float(lof_p99)

if lof_p99 <= lof_p1:
    raise ValueError(
        "Invalid LOF normalization bounds: "
        f"p1={lof_p1}, p99={lof_p99}"
    )

LOF_scores_clipped = np.clip(
    LOF_scores_raw,
    lof_p1,
    lof_p99
)

LOF_norm = (
    LOF_scores_clipped - lof_p1
) / (
    lof_p99 - lof_p1 + 1e-8
)

LOF_norm = LOF_norm.astype(
    np.float32,
    copy=False
)

# ------------------------------------------------------------
# Select exactly the highest-scoring 2%
# ------------------------------------------------------------

(
    LOF_anomalies,
    LOF_threshold,
    LOF_selected_indices
) = select_exact_top_fraction(
    scores=LOF_norm,
    fraction=LOF_TOP_FRACTION
)

lof_selected_count = int(
    LOF_anomalies.sum()
)

lof_expected_count = int(
    np.ceil(
        LOF_TOP_FRACTION * n_samples
    )
)

assert (
    lof_selected_count
    == lof_expected_count
)

# ------------------------------------------------------------
# Save clean LOF results
# ------------------------------------------------------------

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_LOF_scores_raw.npy",
    LOF_scores_raw
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_LOF_scores_norm.npy",
    LOF_norm
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_LOF_anomaly_labels.npy",
    LOF_anomalies
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_LOF_selected_indices.npy",
    LOF_selected_indices.astype(
        np.int64
    )
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_LOF_normalization_params.npz",
    p1=np.float64(
        lof_p1
    ),
    p99=np.float64(
        lof_p99
    )
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_LOF_selection_params.npz",
    fraction=np.float64(
        LOF_TOP_FRACTION
    ),
    selected_count=np.int64(
        lof_selected_count
    ),
    threshold=np.float64(
        LOF_threshold
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nLOF baseline completed.")

print(
    "Raw score range:",
    f"{LOF_scores_raw.min():.6f} to "
    f"{LOF_scores_raw.max():.6f}"
)

print(
    "Normalization bounds:",
    f"p1={lof_p1:.6f}, "
    f"p99={lof_p99:.6f}"
)

print(
    "Selected anomalies:",
    f"{lof_selected_count:,}"
)

print(
    "Expected selected:",
    f"{lof_expected_count:,}"
)

print(
    "Selected percentage:",
    f"{100 * LOF_anomalies.mean():.6f}%"
)

print(
    "Selection threshold:",
    f"{LOF_threshold:.8f}"
)

In [ ]:
# ============================================================
# Clean Mahalanobis Distance baseline
# ============================================================

from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MD_TOP_FRACTION = 0.02
MD_BATCH_SIZE = 100_000
MD_REGULARIZATION = 1e-6

# ------------------------------------------------------------
# Estimate clean mean and covariance from the fixed sample
# ------------------------------------------------------------

mu = np.mean(
    X_sample,
    axis=0,
    dtype=np.float64
)

cov = np.cov(
    X_sample,
    rowvar=False
)

cov = np.asarray(
    cov,
    dtype=np.float64
)

# Regularization for numerical stability
cov += (
    np.eye(
        cov.shape[0],
        dtype=np.float64
    )
    * MD_REGULARIZATION
)

# Pseudoinverse is safer than direct inverse
inv_cov = np.linalg.pinv(
    cov
)

if not np.isfinite(mu).all():
    raise ValueError(
        "Mahalanobis mean vector contains NaN or infinite values."
    )

if not np.isfinite(inv_cov).all():
    raise ValueError(
        "Mahalanobis inverse covariance contains NaN or infinite values."
    )

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_MD_feat_mu.npy",
    mu
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_MD_feat_cov.npy",
    cov
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_MD_feat_inv_cov.npy",
    inv_cov
)

print("Mahalanobis parameters computed and saved.")
print("Mean shape:", mu.shape)
print("Covariance shape:", cov.shape)

# ------------------------------------------------------------
# Compute Mahalanobis scores for the complete clean dataset
# ------------------------------------------------------------

def compute_mahalanobis_scores(
    X_data,
    mean_vector,
    inverse_covariance,
    batch_size=100_000
):
    """
    Compute Mahalanobis distances in batches.

    Higher values indicate greater anomaly likelihood.
    """

    score_batches = []

    total_batches = int(
        np.ceil(
            len(X_data) / batch_size
        )
    )

    for start_idx in tqdm(
        range(0, len(X_data), batch_size),
        total=total_batches,
        desc="Computing MD scores",
        unit="batch"
    ):
        end_idx = min(
            start_idx + batch_size,
            len(X_data)
        )

        batch = np.asarray(
            X_data[start_idx:end_idx],
            dtype=np.float64
        )

        diff = batch - mean_vector

        squared_distances = np.einsum(
            "ij,jk,ik->i",
            diff,
            inverse_covariance,
            diff,
            optimize=True
        )

        # Small negative values can occur due to floating-point error
        squared_distances = np.maximum(
            squared_distances,
            0.0
        )

        batch_scores = np.sqrt(
            squared_distances
        )

        score_batches.append(
            batch_scores.astype(
                np.float32,
                copy=False
            )
        )

    return np.concatenate(
        score_batches
    )


MD_scores_raw = compute_mahalanobis_scores(
    X_data=X_full,
    mean_vector=mu,
    inverse_covariance=inv_cov,
    batch_size=MD_BATCH_SIZE
)

if MD_scores_raw.shape != (n_samples,):
    raise ValueError(
        "Unexpected Mahalanobis score shape: "
        f"{MD_scores_raw.shape}"
    )

if not np.isfinite(
    MD_scores_raw
).all():
    raise ValueError(
        "Mahalanobis scores contain NaN or infinite values."
    )

print(
    "Mahalanobis scores computed:",
    MD_scores_raw.shape
)

# ------------------------------------------------------------
# Robust score normalization
# ------------------------------------------------------------

md_p1, md_p99 = np.percentile(
    MD_scores_raw,
    [1, 99]
)

md_p1 = float(md_p1)
md_p99 = float(md_p99)

if md_p99 <= md_p1:
    raise ValueError(
        "Invalid Mahalanobis normalization bounds: "
        f"p1={md_p1}, p99={md_p99}"
    )

MD_scores_clipped = np.clip(
    MD_scores_raw,
    md_p1,
    md_p99
)

MD_norm = (
    MD_scores_clipped - md_p1
) / (
    md_p99 - md_p1 + 1e-8
)

MD_norm = MD_norm.astype(
    np.float32,
    copy=False
)

# ------------------------------------------------------------
# Select exactly the highest-scoring 2%
# ------------------------------------------------------------

(
    MD_anomalies,
    MD_threshold,
    MD_selected_indices
) = select_exact_top_fraction(
    scores=MD_norm,
    fraction=MD_TOP_FRACTION
)

md_selected_count = int(
    MD_anomalies.sum()
)

md_expected_count = int(
    np.ceil(
        MD_TOP_FRACTION * n_samples
    )
)

assert (
    md_selected_count
    == md_expected_count
)

# ------------------------------------------------------------
# Save clean Mahalanobis results
# ------------------------------------------------------------

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_MD_scores_raw.npy",
    MD_scores_raw
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_MD_scores_norm.npy",
    MD_norm
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_MD_anomaly_labels.npy",
    MD_anomalies
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_MD_selected_indices.npy",
    MD_selected_indices.astype(
        np.int64
    )
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_MD_normalization_params.npz",
    p1=np.float64(
        md_p1
    ),
    p99=np.float64(
        md_p99
    )
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_MD_selection_params.npz",
    fraction=np.float64(
        MD_TOP_FRACTION
    ),
    selected_count=np.int64(
        md_selected_count
    ),
    threshold=np.float64(
        MD_threshold
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nMahalanobis baseline completed.")

print(
    "Raw score range:",
    f"{MD_scores_raw.min():.6f} to "
    f"{MD_scores_raw.max():.6f}"
)

print(
    "Normalization bounds:",
    f"p1={md_p1:.6f}, "
    f"p99={md_p99:.6f}"
)

print(
    "Normalized score statistics:"
)

print(
    f"  min={MD_norm.min():.6f}"
)

print(
    f"  max={MD_norm.max():.6f}"
)

print(
    f"  mean={MD_norm.mean():.6f}"
)

print(
    "Selected anomalies:",
    f"{md_selected_count:,}"
)

print(
    "Expected selected:",
    f"{md_expected_count:,}"
)

print(
    "Selected percentage:",
    f"{100 * MD_anomalies.mean():.6f}%"
)

print(
    "Selection threshold:",
    f"{MD_threshold:.8f}"
)

In [ ]:
# ============================================================
# Agreement analysis among IF, LOF, and MD baselines
# ============================================================

# Binary union: selected by at least one method
baseline_union = (
    IF_anomalies.astype(bool)
    | LOF_anomalies.astype(bool)
    | MD_anomalies.astype(bool)
)

# Binary intersection: selected by all three methods
baseline_intersection = (
    IF_anomalies.astype(bool)
    & LOF_anomalies.astype(bool)
    & MD_anomalies.astype(bool)
)

# Agreement count:
# 0 = selected by no method
# 1 = selected by one method
# 2 = selected by two methods
# 3 = selected by all three methods
baseline_agreement = (
    IF_anomalies.astype(np.uint8)
    + LOF_anomalies.astype(np.uint8)
    + MD_anomalies.astype(np.uint8)
)

agreement_values, agreement_counts = np.unique(
    baseline_agreement,
    return_counts=True
)

print("Baseline agreement distribution:")

for value, count in zip(
    agreement_values,
    agreement_counts
):
    print(
        f"Agreement = {value}: "
        f"{count:,} pixels "
        f"({100 * count / n_samples:.4f}%)"
    )

print(
    "\nUnion count:",
    f"{baseline_union.sum():,}",
    f"({100 * baseline_union.mean():.4f}%)"
)

print(
    "Intersection count:",
    f"{baseline_intersection.sum():,}",
    f"({100 * baseline_intersection.mean():.4f}%)"
)

# Save diagnostic outputs
np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_IF_LOF_MD_union.npy",
    baseline_union.astype(np.uint8)
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_IF_LOF_MD_intersection.npy",
    baseline_intersection.astype(np.uint8)
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_IF_LOF_MD_agreement.npy",
    baseline_agreement
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_IF_LOF_MD_agreement_summary.npz",
    agreement_values=agreement_values,
    agreement_counts=agreement_counts,
    union_count=np.int64(
        baseline_union.sum()
    ),
    intersection_count=np.int64(
        baseline_intersection.sum()
    )
)

In [ ]:
# ============================================================
# Clean PCA reconstruction baseline
# ============================================================

import joblib
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

PCA_TOP_FRACTION = 0.02
PCA_BATCH_SIZE = 100_000
PCA_VARIANCE_RATIO = 0.95

# ------------------------------------------------------------
# Fit PCA on the same clean fitting subset
# ------------------------------------------------------------

pca_model = PCA(
    n_components=PCA_VARIANCE_RATIO,
    svd_solver="full",
    random_state=7
)

pca_model.fit(
    X_sample
)

joblib.dump(
    pca_model,
    BASELINE_RESULTS_DIR
    / "Baseline_PCA_model.pkl"
)

print("PCA fitted and saved.")
print("Retained components:", pca_model.n_components_)
print(
    "Explained variance:",
    f"{pca_model.explained_variance_ratio_.sum():.6f}"
)

# ------------------------------------------------------------
# Compute full-data reconstruction errors in batches
# ------------------------------------------------------------

def compute_pca_reconstruction_errors(
    model,
    X_data,
    batch_size=100_000
):
    """
    Compute per-sample PCA reconstruction MSE in batches.

    Higher values indicate greater anomaly likelihood.
    """

    error_batches = []

    total_batches = int(
        np.ceil(
            len(X_data) / batch_size
        )
    )

    for start_idx in tqdm(
        range(0, len(X_data), batch_size),
        total=total_batches,
        desc="Computing PCA errors",
        unit="batch"
    ):
        end_idx = min(
            start_idx + batch_size,
            len(X_data)
        )

        batch = np.asarray(
            X_data[start_idx:end_idx],
            dtype=np.float32
        )

        projected = model.transform(
            batch
        )

        reconstructed = model.inverse_transform(
            projected
        )

        batch_errors = np.mean(
            (
                batch - reconstructed
            ) ** 2,
            axis=1
        )

        error_batches.append(
            batch_errors.astype(
                np.float32,
                copy=False
            )
        )

    return np.concatenate(
        error_batches
    )


PCA_scores_raw = compute_pca_reconstruction_errors(
    model=pca_model,
    X_data=X_full,
    batch_size=PCA_BATCH_SIZE
)

if PCA_scores_raw.shape != (n_samples,):
    raise ValueError(
        "Unexpected PCA score shape: "
        f"{PCA_scores_raw.shape}"
    )

if not np.isfinite(
    PCA_scores_raw
).all():
    raise ValueError(
        "PCA reconstruction errors contain NaN or Inf."
    )

print(
    "PCA reconstruction errors computed:",
    PCA_scores_raw.shape
)

# ------------------------------------------------------------
# Robust score normalization
# ------------------------------------------------------------

pca_p1, pca_p99 = np.percentile(
    PCA_scores_raw,
    [1, 99]
)

pca_p1 = float(pca_p1)
pca_p99 = float(pca_p99)

if pca_p99 <= pca_p1:
    raise ValueError(
        "Invalid PCA normalization bounds: "
        f"p1={pca_p1}, p99={pca_p99}"
    )

PCA_scores_clipped = np.clip(
    PCA_scores_raw,
    pca_p1,
    pca_p99
)

PCA_norm = (
    PCA_scores_clipped - pca_p1
) / (
    pca_p99 - pca_p1 + 1e-8
)

PCA_norm = PCA_norm.astype(
    np.float32,
    copy=False
)

# ------------------------------------------------------------
# Select exactly the highest-scoring 2%
# ------------------------------------------------------------

(
    PCA_anomalies,
    PCA_threshold,
    PCA_selected_indices
) = select_exact_top_fraction(
    scores=PCA_norm,
    fraction=PCA_TOP_FRACTION
)

pca_selected_count = int(
    PCA_anomalies.sum()
)

pca_expected_count = int(
    np.ceil(
        PCA_TOP_FRACTION * n_samples
    )
)

assert pca_selected_count == pca_expected_count

# ------------------------------------------------------------
# Save PCA artifacts and outputs
# ------------------------------------------------------------

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_PCA_scores_raw.npy",
    PCA_scores_raw
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_PCA_scores_norm.npy",
    PCA_norm
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_PCA_anomaly_labels.npy",
    PCA_anomalies
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_PCA_selected_indices.npy",
    PCA_selected_indices.astype(np.int64)
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_PCA_normalization_params.npz",
    p1=np.float64(pca_p1),
    p99=np.float64(pca_p99)
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_PCA_selection_params.npz",
    fraction=np.float64(
        PCA_TOP_FRACTION
    ),
    selected_count=np.int64(
        pca_selected_count
    ),
    threshold=np.float64(
        PCA_threshold
    ),
    retained_components=np.int64(
        pca_model.n_components_
    ),
    explained_variance=np.float64(
        pca_model.explained_variance_ratio_.sum()
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nPCA baseline completed.")

print(
    "Raw error range:",
    f"{PCA_scores_raw.min():.8f} to "
    f"{PCA_scores_raw.max():.8f}"
)

print(
    "Normalization bounds:",
    f"p1={pca_p1:.8f}, "
    f"p99={pca_p99:.8f}"
)

print(
    "Normalized mean:",
    f"{PCA_norm.mean():.6f}"
)

print(
    "Normalized 95th percentile:",
    f"{np.percentile(PCA_norm, 95):.6f}"
)

print(
    "Normalized 98th percentile:",
    f"{np.percentile(PCA_norm, 98):.6f}"
)

print(
    "Selected anomalies:",
    f"{pca_selected_count:,}"
)

print(
    "Selected percentage:",
    f"{100 * PCA_anomalies.mean():.6f}%"
)

In [ ]:
# ============================================================
# Clean conventional autoencoder baseline
# ============================================================

import random

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# SEED is already defined in the initial setup cell
# ------------------------------------------------------------

AE_EPOCHS = 10
AE_TRAIN_BATCH_SIZE = 256
AE_INFERENCE_BATCH_SIZE = 4_096
AE_LEARNING_RATE = 1e-3
AE_TOP_FRACTION = 0.02

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Improve reproducibility on supported backends
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Generator used only for deterministic DataLoader shuffling
ae_loader_generator = torch.Generator()
ae_loader_generator.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("AE device:", device)
print("AE seed:", SEED)

# ------------------------------------------------------------
# Prepare the same clean training subset
# ------------------------------------------------------------

X_train_tensor = torch.from_numpy(
    np.asarray(
        X_sample,
        dtype=np.float32
    )
)

train_dataset = TensorDataset(
    X_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=AE_TRAIN_BATCH_SIZE,
    shuffle=True,
    generator=ae_loader_generator,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# ------------------------------------------------------------
# Define conventional single-view autoencoder
# ------------------------------------------------------------

input_dim = X_sample.shape[1]


class BaselineAutoEncoder(nn.Module):

    def __init__(self, input_dimension):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dimension, 64),
            nn.ReLU(),
            nn.Linear(64, 16)
        )

        self.decoder = nn.Sequential(
            nn.Linear(16, 64),
            nn.ReLU(),
            nn.Linear(64, input_dimension)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        return reconstruction


ae_model = BaselineAutoEncoder(
    input_dimension=input_dim
).to(device)

optimizer = torch.optim.Adam(
    ae_model.parameters(),
    lr=AE_LEARNING_RATE
)

criterion = nn.MSELoss(
    reduction="mean"
)

# ------------------------------------------------------------
# Train the autoencoder
# ------------------------------------------------------------

training_history = []

for epoch in range(AE_EPOCHS):

    ae_model.train()

    epoch_loss_sum = 0.0
    epoch_sample_count = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"AE epoch {epoch + 1}/{AE_EPOCHS}",
        unit="batch"
    )

    for batch_tuple in progress_bar:

        x_batch = batch_tuple[0].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        reconstructed = ae_model(
            x_batch
        )

        loss = criterion(
            reconstructed,
            x_batch
        )

        loss.backward()
        optimizer.step()

        current_batch_size = x_batch.size(0)

        epoch_loss_sum += (
            loss.item() * current_batch_size
        )

        epoch_sample_count += current_batch_size

        progress_bar.set_postfix(
            loss=f"{loss.item():.6f}"
        )

    epoch_mean_loss = (
        epoch_loss_sum
        / epoch_sample_count
    )

    training_history.append(
        epoch_mean_loss
    )

    print(
        f"Epoch {epoch + 1}: "
        f"mean loss = {epoch_mean_loss:.8f}"
    )

# ------------------------------------------------------------
# Save the trained autoencoder
# ------------------------------------------------------------

torch.save(
    {
        "model_state_dict": ae_model.state_dict(),
        "input_dim": input_dim,
        "hidden_dim": 64,
        "latent_dim": 16,
        "epochs": AE_EPOCHS,
        "train_batch_size": AE_TRAIN_BATCH_SIZE,
        "learning_rate": AE_LEARNING_RATE,
        "seed": SEED,
        "training_history": training_history
    },
    BASELINE_RESULTS_DIR
    / "Baseline_AE_model.pth"
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_AE_training_history.npy",
    np.asarray(
        training_history,
        dtype=np.float64
    )
)

print("Autoencoder trained and saved.")

# ------------------------------------------------------------
# Compute full clean-data reconstruction errors
# ------------------------------------------------------------

def compute_ae_reconstruction_errors(
    model,
    X_data,
    device,
    batch_size=4_096
):
    """
    Compute per-sample autoencoder reconstruction MSE.

    Higher values indicate greater anomaly likelihood.
    """

    inference_tensor = torch.from_numpy(
        np.asarray(
            X_data,
            dtype=np.float32
        )
    )

    inference_loader = DataLoader(
        inference_tensor,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda")
    )

    error_batches = []

    model.eval()

    with torch.no_grad():

        for x_batch in tqdm(
            inference_loader,
            desc="Computing AE errors",
            unit="batch"
        ):

            x_batch = x_batch.to(
                device,
                non_blocking=True
            )

            reconstructed = model(
                x_batch
            )

            batch_errors = torch.mean(
                (x_batch - reconstructed) ** 2,
                dim=1
            )

            error_batches.append(
                batch_errors
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False
                )
            )

    return np.concatenate(
        error_batches
    )


AE_scores_raw = compute_ae_reconstruction_errors(
    model=ae_model,
    X_data=X_full,
    device=device,
    batch_size=AE_INFERENCE_BATCH_SIZE
)

if AE_scores_raw.shape != (n_samples,):
    raise ValueError(
        "Unexpected AE score shape: "
        f"{AE_scores_raw.shape}"
    )

if not np.isfinite(AE_scores_raw).all():
    raise ValueError(
        "AE reconstruction errors contain NaN or infinite values."
    )

print(
    "AE reconstruction errors computed:",
    AE_scores_raw.shape
)

# ------------------------------------------------------------
# Robust score normalization
# ------------------------------------------------------------

ae_p1, ae_p99 = np.percentile(
    AE_scores_raw,
    [1, 99]
)

ae_p1 = float(ae_p1)
ae_p99 = float(ae_p99)

if ae_p99 <= ae_p1:
    raise ValueError(
        "Invalid AE normalization bounds: "
        f"p1={ae_p1}, p99={ae_p99}"
    )

AE_scores_clipped = np.clip(
    AE_scores_raw,
    ae_p1,
    ae_p99
)

AE_norm = (
    AE_scores_clipped - ae_p1
) / (
    ae_p99 - ae_p1 + 1e-8
)

AE_norm = AE_norm.astype(
    np.float32,
    copy=False
)

# ------------------------------------------------------------
# Select exactly the highest-scoring 2%
# ------------------------------------------------------------

(
    AE_anomalies,
    AE_threshold,
    AE_selected_indices
) = select_exact_top_fraction(
    scores=AE_norm,
    fraction=AE_TOP_FRACTION
)

ae_selected_count = int(
    AE_anomalies.sum()
)

ae_expected_count = int(
    np.ceil(
        AE_TOP_FRACTION * n_samples
    )
)

assert ae_selected_count == ae_expected_count

# ------------------------------------------------------------
# Save clean AE outputs
# ------------------------------------------------------------

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_AE_scores_raw.npy",
    AE_scores_raw
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_AE_scores_norm.npy",
    AE_norm
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_AE_anomaly_labels.npy",
    AE_anomalies
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_AE_selected_indices.npy",
    AE_selected_indices.astype(
        np.int64
    )
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_AE_normalization_params.npz",
    p1=np.float64(ae_p1),
    p99=np.float64(ae_p99)
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_AE_selection_params.npz",
    fraction=np.float64(
        AE_TOP_FRACTION
    ),
    selected_count=np.int64(
        ae_selected_count
    ),
    threshold=np.float64(
        AE_threshold
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nAutoencoder baseline completed.")

print(
    "Raw error range:",
    f"{AE_scores_raw.min():.8f} to "
    f"{AE_scores_raw.max():.8f}"
)

print(
    "Normalization bounds:",
    f"p1={ae_p1:.8f}, "
    f"p99={ae_p99:.8f}"
)

print(
    "Normalized mean:",
    f"{AE_norm.mean():.6f}"
)

print(
    "Normalized 95th percentile:",
    f"{np.percentile(AE_norm, 95):.6f}"
)

print(
    "Normalized 98th percentile:",
    f"{np.percentile(AE_norm, 98):.6f}"
)

print(
    "Selected anomalies:",
    f"{ae_selected_count:,}"
)

print(
    "Selected percentage:",
    f"{100 * AE_anomalies.mean():.6f}%"
)

print(
    "Selection threshold:",
    f"{AE_threshold:.8f}"
)

In [ ]:
# ============================================================
# Clean Histogram-Based Outlier Score baseline
# ============================================================

import joblib
from pyod.models.hbos import HBOS
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

HBOS_N_BINS = 10
HBOS_TOP_FRACTION = 0.02
HBOS_BATCH_SIZE = 100_000

# ------------------------------------------------------------
# Train HBOS on the same clean fitting subset
# ------------------------------------------------------------

hbos_model = HBOS(
    n_bins=HBOS_N_BINS
)

hbos_model.fit(
    X_sample
)

joblib.dump(
    hbos_model,
    BASELINE_RESULTS_DIR
    / "Baseline_HBOS_model.pkl"
)

print("HBOS trained and saved.")

# ------------------------------------------------------------
# Compute HBOS scores for the complete clean dataset
# ------------------------------------------------------------

def compute_hbos_scores(
    model,
    X_data,
    batch_size=100_000
):
    """
    Compute HBOS anomaly scores in batches.

    Larger scores indicate greater anomaly likelihood.
    """

    score_batches = []

    total_batches = int(
        np.ceil(
            len(X_data) / batch_size
        )
    )

    for start_idx in tqdm(
        range(0, len(X_data), batch_size),
        total=total_batches,
        desc="Computing HBOS scores",
        unit="batch"
    ):
        end_idx = min(
            start_idx + batch_size,
            len(X_data)
        )

        batch = np.asarray(
            X_data[start_idx:end_idx],
            dtype=np.float32
        )

        batch_scores = model.decision_function(
            batch
        )

        score_batches.append(
            batch_scores.astype(
                np.float32,
                copy=False
            )
        )

    return np.concatenate(
        score_batches
    )


HBOS_scores_raw = compute_hbos_scores(
    model=hbos_model,
    X_data=X_full,
    batch_size=HBOS_BATCH_SIZE
)

if HBOS_scores_raw.shape != (n_samples,):
    raise ValueError(
        "Unexpected HBOS score shape: "
        f"{HBOS_scores_raw.shape}"
    )

if not np.isfinite(
    HBOS_scores_raw
).all():
    raise ValueError(
        "HBOS scores contain NaN or infinite values."
    )

print(
    "HBOS scores computed:",
    HBOS_scores_raw.shape
)

# ------------------------------------------------------------
# Robust score normalization
# ------------------------------------------------------------

hbos_p1, hbos_p99 = np.percentile(
    HBOS_scores_raw,
    [1, 99]
)

hbos_p1 = float(hbos_p1)
hbos_p99 = float(hbos_p99)

if hbos_p99 <= hbos_p1:
    raise ValueError(
        "Invalid HBOS normalization bounds: "
        f"p1={hbos_p1}, p99={hbos_p99}"
    )

HBOS_scores_clipped = np.clip(
    HBOS_scores_raw,
    hbos_p1,
    hbos_p99
)

HBOS_norm = (
    HBOS_scores_clipped - hbos_p1
) / (
    hbos_p99 - hbos_p1 + 1e-8
)

HBOS_norm = HBOS_norm.astype(
    np.float32,
    copy=False
)

# ------------------------------------------------------------
# Select exactly the highest-scoring 2%
# ------------------------------------------------------------

(
    HBOS_anomalies,
    HBOS_threshold,
    HBOS_selected_indices
) = select_exact_top_fraction(
    scores=HBOS_norm,
    fraction=HBOS_TOP_FRACTION
)

hbos_selected_count = int(
    HBOS_anomalies.sum()
)

hbos_expected_count = int(
    np.ceil(
        HBOS_TOP_FRACTION * n_samples
    )
)

assert hbos_selected_count == hbos_expected_count

# ------------------------------------------------------------
# Save clean HBOS artifacts and outputs
# ------------------------------------------------------------

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_HBOS_scores_raw.npy",
    HBOS_scores_raw
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_HBOS_scores_norm.npy",
    HBOS_norm
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_HBOS_anomaly_labels.npy",
    HBOS_anomalies
)

np.save(
    BASELINE_RESULTS_DIR
    / "Baseline_HBOS_selected_indices.npy",
    HBOS_selected_indices.astype(
        np.int64
    )
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_HBOS_normalization_params.npz",
    p1=np.float64(
        hbos_p1
    ),
    p99=np.float64(
        hbos_p99
    )
)

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_HBOS_selection_params.npz",
    fraction=np.float64(
        HBOS_TOP_FRACTION
    ),
    selected_count=np.int64(
        hbos_selected_count
    ),
    threshold=np.float64(
        HBOS_threshold
    ),
    n_bins=np.int64(
        HBOS_N_BINS
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nHBOS baseline completed.")

print(
    "Raw score range:",
    f"{HBOS_scores_raw.min():.8f} to "
    f"{HBOS_scores_raw.max():.8f}"
)

print(
    "Normalization bounds:",
    f"p1={hbos_p1:.8f}, "
    f"p99={hbos_p99:.8f}"
)

print(
    "Normalized mean:",
    f"{HBOS_norm.mean():.6f}"
)

print(
    "Normalized 95th percentile:",
    f"{np.percentile(HBOS_norm, 95):.6f}"
)

print(
    "Normalized 98th percentile:",
    f"{np.percentile(HBOS_norm, 98):.6f}"
)

print(
    "Selected anomalies:",
    f"{hbos_selected_count:,}"
)

print(
    "Expected selected:",
    f"{hbos_expected_count:,}"
)

print(
    "Selected percentage:",
    f"{100 * HBOS_anomalies.mean():.6f}%"
)

print(
    "Selection threshold:",
    f"{HBOS_threshold:.8f}"
)

In [ ]:
# ============================================================
# Flood-reference overlap of clean baseline detections
# ============================================================

# y was loaded in the initial setup cell
flood_mask = np.asarray(
    y > 0,
    dtype=bool
)

if flood_mask.shape != (n_samples,):
    raise ValueError(
        "Unexpected flood-mask shape: "
        f"{flood_mask.shape}"
    )


def compute_reference_overlap(
    anomaly_labels,
    reference_mask
):
    """
    Compute the proportion of selected anomaly candidates
    located inside the reference flood mask.

    This is an overlap measure, not precision, because the
    flood-risk map is not complete anomaly ground truth.
    """

    anomaly_labels = np.asarray(
        anomaly_labels,
        dtype=bool
    )

    selected_count = int(
        anomaly_labels.sum()
    )

    if selected_count == 0:
        return np.nan

    overlap_count = int(
        np.sum(
            anomaly_labels
            & reference_mask
        )
    )

    overlap_ratio = (
        overlap_count
        / selected_count
    )

    return overlap_ratio


baseline_labels = {
    "IF": IF_anomalies,
    "LOF": LOF_anomalies,
    "MD": MD_anomalies,
    "PCA": PCA_anomalies,
    "AE": AE_anomalies,
    "HBOS": HBOS_anomalies
}

print("=== Flood-reference overlap of clean baseline candidates ===")

flood_overlap_results = {}

for method_name, method_labels in baseline_labels.items():

    overlap_ratio = compute_reference_overlap(
        anomaly_labels=method_labels,
        reference_mask=flood_mask
    )

    flood_overlap_results[
        method_name
    ] = overlap_ratio

    print(
        f"{method_name}: "
        f"{100 * overlap_ratio:.4f}%"
    )

np.savez(
    BASELINE_RESULTS_DIR
    / "Baseline_flood_reference_overlap.npz",
    **{
        method_name: np.float64(overlap_ratio)
        for method_name, overlap_ratio
        in flood_overlap_results.items()
    }
)

In [ ]:
# ============================================================
# Final clean-data consistency checks
# ============================================================

print("Full scaled dataset shape:", X_full.shape)
print("Fitting subset shape:", X_sample.shape)

print("\nFull scaled dataset statistics:")
print("Mean:", float(np.mean(X_full)))
print("Standard deviation:", float(np.std(X_full)))
print("Minimum:", float(np.min(X_full)))
print("Maximum:", float(np.max(X_full)))

print("\nFixed fitting subset statistics:")
print("Mean:", float(np.mean(X_sample)))
print("Standard deviation:", float(np.std(X_sample)))
print("Minimum:", float(np.min(X_sample)))
print("Maximum:", float(np.max(X_sample)))

print(
    "\nSaved fitting indices:",
    sample_indices.shape
)

print(
    "Unique fitting indices:",
    np.unique(sample_indices).size
)

assert len(sample_indices) == len(X_sample)
assert np.unique(sample_indices).size == len(sample_indices)
assert np.array_equal(
    X_sample,
    X_full[sample_indices]
)

print("\nBaseline fitting subset consistency check passed.")